## 07 - Knowledge Base and Agent Prompts

### Objetivo
Resolver:

- 6. Construcción de una pequeña base de conocimiento para productos líderes.
- 7. Diseño de 3 system prompts para un agente hiper-personalizado.

### Decisiones metodológicas
- La base de conocimiento partirá de los productos más vendidos identificados en Q1.
- Como el dataset no contiene nombre comercial del producto, la unidad de referencia será:
  - `product_id`
  - `product_category`
- El precio incluido será el **precio real promedio** calculado desde `order_items.price`.
- Los prompts usarán:
  - `[EDAD]`
  - `[GÉNERO]`
  - `[PERFIL_DE_CLIENTE]`
  - `[HISTORIAL_COMPRAS]`
  - `[BASE_CONOCIMIENTO]`

In [4]:
from pathlib import Path
import sys
import json
from pprint import pprint

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_all_datasets
from src.analysis.knowledge_base import (
    build_item_level_sales_base,
    compute_product_sales,
    compute_top_products,
    select_reference_products,
    build_knowledge_base_dict,
    knowledge_base_to_json,
    save_knowledge_base_json,
)
from src.prompts.system_prompts import (
    SYSTEM_PROMPTS,
    build_purchase_history_summary,
    render_system_prompt,
)

In [5]:
processed_dir = PROJECT_ROOT / "data" / "processed"

customer_360_features = pd.read_csv(processed_dir / "customer_360_features.csv")

bundle = load_all_datasets()

In [8]:
## Reconstrucción de productos líderes a partir de Q1

item_base = build_item_level_sales_base(bundle)
product_sales = compute_product_sales(item_base)

top_products_volume, top_products_revenue = compute_top_products(product_sales, top_k=5)

top_products_volume

,product_id,product_category,total_volume,total_revenue,avg_price,n_orders
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,37104.30,71.354423,425
1,422879e10f46682990de24d770e7f83d,garden_tools,484,26577.22,54.911612,352
2,99a4788cb24856965c36a24e339b6058,bed_bath_table,477,42049.66,88.154423,456
3,389d119b48cf3043d311335e499d9c6b,garden_tools,390,21336.79,54.709718,309
4,368c6c730842d78016ad823897a372db,garden_tools,388,21056.80,54.270103,291


In [7]:
top_products_revenue

,product_id,product_category,total_volume,total_revenue,avg_price,n_orders
0,bb50f2e236e5eea0100680137654686c,health_beauty,194,63560.00,327.628866,186
1,6cdd53843498f92890544667809f1595,health_beauty,153,53652.30,350.668627,148
2,d6160fb7873f184099d9bc95e30376af,computers,33,45949.35,1392.404545,33
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,45620.56,137.411325,313
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,477,42049.66,88.154423,456


### 1. Selección de 3 productos de referencia

Se seleccionan 3 productos a partir de la unión entre:
- top productos por volumen
- top productos por ingresos

Esto permite construir una base pequeña pero estratégicamente útil.

In [9]:
reference_products = select_reference_products(
    product_sales,
    top_k=5,
    n_products=3,
)

reference_products

,product_id,product_category,total_volume,total_revenue,avg_price,n_orders,rank_volume,rank_revenue,selection_score
0,bb50f2e236e5eea0100680137654686c,health_beauty,194,63560.0,327.628866,186,7.0,1.0,1.142857
1,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,37104.3,71.354423,425,1.0,7.0,1.142857
2,6cdd53843498f92890544667809f1595,health_beauty,153,53652.3,350.668627,148,7.0,2.0,0.642857


### 2. Base de conocimiento simulada

In [10]:
knowledge_base = build_knowledge_base_dict(reference_products)
pprint(knowledge_base)

{'6cdd53843498f92890544667809f1595': {'avg_real_price': 350.67,
                                      'marketing_description': 'Bienestar con '
                                                               'alta tracción '
                                                               'comercial. '
                                                               'Referencia muy '
                                                               'atractiva '
                                                               'dentro del '
                                                               'portafolio de '
                                                               'cuidado '
                                                               'personal, '
                                                               'ideal para '
                                                               'campañas de '
                                                               'recompra, '
    

In [12]:
knowledge_base_json = knowledge_base_to_json(knowledge_base)
print(knowledge_base_json)

{
  "bb50f2e236e5eea0100680137654686c": {
    "product_id": "bb50f2e236e5eea0100680137654686c",
    "product_category": "health_beauty",
    "avg_real_price": 327.63,
    "sales_signals": {
      "total_volume": 194,
      "total_revenue": 63560.0,
      "n_orders": 186
    },
    "marketing_description": "Bienestar con alta tracción comercial. Referencia muy atractiva dentro del portafolio de cuidado personal, ideal para campañas de recompra, rutina y autocuidado. En el dataset mostró una tracción destacada con 194 ítems vendidos y 186 órdenes asociadas. Su precio real promedio fue de $327.63, lo que lo convierte en una referencia atractiva para campañas de recomendación y personalización."
  },
  "aca2eb7d00ea1a7b8ebd4e68314663af": {
    "product_id": "aca2eb7d00ea1a7b8ebd4e68314663af",
    "product_category": "furniture_decor",
    "avg_real_price": 71.35,
    "sales_signals": {
      "total_volume": 520,
      "total_revenue": 37104.3,
      "n_orders": 425
    },
    "marketing_de

In [13]:
output_path = processed_dir / "product_knowledge_base.json"
save_knowledge_base_json(knowledge_base, str(output_path))

print("Knowledge base exportada en:", output_path)

Knowledge base exportada en: D:\Users\dhcertug\OneDrive - Crystal S.A.S\Escritorio\commercial-analytics-ai-challenge\data\processed\product_knowledge_base.json


### 3. Conclusión de Q6

La base de conocimiento quedó construida sobre productos líderes del negocio, usando:
- identificador real (`product_id`)
- categoría del producto
- precio promedio real
- señales de volumen e ingresos
- descripción comercial redactada de forma atractiva y reusable


## 4. Preparación de contexto para prompts del agente

Se construirán 3 escenarios:
1. Cliente joven, digital y comprador frecuente.
2. Cliente mayor, conservador/inactivo, con mala experiencia previa.
3. Cliente VIP o de alto gasto.

In [14]:
scenario_1_row = (
    customer_360_features
    .query("customer_segment_primary in ['CORE_LOYAL', 'VIP']")
    .sort_values(["frequency", "monetary"], ascending=[False, False])
    .iloc[0]
)

scenario_2_candidates = customer_360_features.loc[
    customer_360_features["customer_flags_text"].fillna("").str.contains("FRICTION_RISK", regex=False)
].copy()

if scenario_2_candidates.empty:
    scenario_2_row = (
        customer_360_features
        .sort_values(["recency_days", "frequency"], ascending=[False, True])
        .iloc[0]
    )
else:
    scenario_2_row = (
        scenario_2_candidates
        .sort_values(["recency_days", "inactivity_risk_score"], ascending=[False, False])
        .iloc[0]
    )

scenario_3_row = (
    customer_360_features
    .query("customer_segment_primary == 'VIP'")
    .sort_values(["monetary", "avg_order_value"], ascending=[False, False])
    .iloc[0]
)

In [15]:
scenario_context = pd.DataFrame([
    {
        "scenario": "young_digital_frequent",
        "edad": 24,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_1_row["customer_segment_primary"],
        "historial_compras": build_purchase_history_summary(scenario_1_row.to_dict()),
        "customer_unique_id": scenario_1_row["customer_unique_id"],
    },
    {
        "scenario": "older_conservative_friction",
        "edad": 63,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_2_row["customer_segment_primary"],
        "historial_compras": build_purchase_history_summary(scenario_2_row.to_dict()),
        "customer_unique_id": scenario_2_row["customer_unique_id"],
    },
    {
        "scenario": "vip_high_spend",
        "edad": 42,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_3_row["customer_segment_primary"],
        "historial_compras": build_purchase_history_summary(scenario_3_row.to_dict()),
        "customer_unique_id": scenario_3_row["customer_unique_id"],
    },
])

scenario_context

,scenario,edad,genero,perfil_de_cliente,historial_compras,customer_unique_id
0,young_digital_frequent,24,No especificado,VIP,"Cliente con compra recurrente, última activida...",8d50f5eadf50201ccdcedfb9e2ac8455
1,older_conservative_friction,63,No especificado,HIGH_VALUE_RISK,"Cliente con compra esporádica, última activida...",830d5b7aaa3b6f1e9ad63703bec97d23
2,vip_high_spend,42,No especificado,VIP,"Cliente con compra recurrente, última activida...",c8460e4251689ba205045f3ea17884a1


### 5. Templates de system prompts

In [16]:
list(SYSTEM_PROMPTS.keys())

['young_digital_frequent', 'older_conservative_friction', 'vip_high_spend']

In [17]:
print(SYSTEM_PROMPTS["young_digital_frequent"])

Eres el system prompt de un agente de ecommerce para clientes jóvenes, digitales y frecuentes.

Tu misión es recomendar productos de forma hiper-personalizada, con tono ágil, cercano, actual y orientado a conveniencia.
Habla en español claro, natural y breve. Evita sonar robótico. No uses lenguaje demasiado técnico.

Contexto dinámico del cliente:
- Edad: [EDAD]
- Género: [GÉNERO]
- Perfil de cliente: [PERFIL_DE_CLIENTE]
- Historial de compras resumido: [HISTORIAL_COMPRAS]
- Base de conocimiento disponible: [BASE_CONOCIMIENTO]

Instrucciones:
1. Usa el perfil y el historial para inferir intereses probables.
2. Prioriza recomendaciones prácticas, actuales y fáciles de convertir.
3. Si el historial muestra afinidad clara por una categoría, empieza por ahí.
4. Ofrece entre 2 y 4 sugerencias, no más.
5. Justifica cada sugerencia con una razón breve y clara.
6. Usa únicamente información consistente con la base de conocimiento.
7. Si el dataset solo identifica productos por product_id, pued

In [18]:
print(SYSTEM_PROMPTS["older_conservative_friction"])

Eres el system prompt de un agente de ecommerce para clientes mayores, conservadores o inactivos, especialmente si tuvieron una mala experiencia previa.

Tu misión es recuperar confianza antes de vender. Habla con tono respetuoso, calmado, empático y seguro.
Debes priorizar claridad, acompañamiento y baja fricción. Evita entusiasmo excesivo y evita sonar agresivo en ventas.

Contexto dinámico del cliente:
- Edad: [EDAD]
- Género: [GÉNERO]
- Perfil de cliente: [PERFIL_DE_CLIENTE]
- Historial de compras resumido: [HISTORIAL_COMPRAS]
- Base de conocimiento disponible: [BASE_CONOCIMIENTO]

Instrucciones:
1. Reconoce de forma sutil que la experiencia del cliente debe ser cuidada.
2. Prioriza recomendaciones conservadoras, claras y coherentes con su historial.
3. Evita mostrar demasiadas opciones; entrega entre 2 y 3.
4. Enfatiza confianza, utilidad, claridad de compra y tranquilidad.
5. No prometas atributos que no estén en la base de conocimiento.
6. Si el cliente muestra señales de fricci

In [19]:
print(SYSTEM_PROMPTS["vip_high_spend"])

Eres el system prompt de un agente de ecommerce para clientes corporativos, premium o de alto gasto (VIP).

Tu misión es recomendar productos con un tono consultivo, ejecutivo y eficiente.
Debes sonar seguro, premium y orientado a valor, no a descuentos masivos.
Habla en español profesional, claro y directo.

Contexto dinámico del cliente:
- Edad: [EDAD]
- Género: [GÉNERO]
- Perfil de cliente: [PERFIL_DE_CLIENTE]
- Historial de compras resumido: [HISTORIAL_COMPRAS]
- Base de conocimiento disponible: [BASE_CONOCIMIENTO]

Instrucciones:
1. Prioriza productos con mayor potencial de valor percibido y afinidad histórica.
2. Presenta las recomendaciones como selección curada.
3. Entrega entre 3 y 5 opciones.
4. Justifica cada sugerencia por relevancia, categoría y nivel de valor.
5. Si aplica, sugiere compra complementaria o bundle lógico.
6. No inventes especificaciones técnicas que no existan en la base de conocimiento.
7. Si el catálogo no tiene nombres comerciales, usa categoría + produc

In [20]:
## Render de prompts finales con variables dinámicas

rendered_prompt_1 = render_system_prompt(
    SYSTEM_PROMPTS["young_digital_frequent"],
    edad=scenario_context.loc[0, "edad"],
    genero=scenario_context.loc[0, "genero"],
    perfil_de_cliente=scenario_context.loc[0, "perfil_de_cliente"],
    historial_compras=scenario_context.loc[0, "historial_compras"],
    base_conocimiento=knowledge_base,
)

print(rendered_prompt_1)

Eres el system prompt de un agente de ecommerce para clientes jóvenes, digitales y frecuentes.

Tu misión es recomendar productos de forma hiper-personalizada, con tono ágil, cercano, actual y orientado a conveniencia.
Habla en español claro, natural y breve. Evita sonar robótico. No uses lenguaje demasiado técnico.

Contexto dinámico del cliente:
- Edad: 24
- Género: No especificado
- Perfil de cliente: VIP
- Historial de compras resumido: Cliente con compra recurrente, última actividad hace 9 días, afinidad principal por la categoría sports_leisure, gasto histórico aproximado de $714.63, ticket promedio de $47.64 y flags actuales: HIGH_VALUE, FRICTION_RISK.
- Base de conocimiento disponible: {
  "bb50f2e236e5eea0100680137654686c": {
    "product_id": "bb50f2e236e5eea0100680137654686c",
    "product_category": "health_beauty",
    "avg_real_price": 327.63,
    "sales_signals": {
      "total_volume": 194,
      "total_revenue": 63560.0,
      "n_orders": 186
    },
    "marketing_desc

In [21]:
rendered_prompt_2 = render_system_prompt(
    SYSTEM_PROMPTS["older_conservative_friction"],
    edad=scenario_context.loc[1, "edad"],
    genero=scenario_context.loc[1, "genero"],
    perfil_de_cliente=scenario_context.loc[1, "perfil_de_cliente"],
    historial_compras=scenario_context.loc[1, "historial_compras"],
    base_conocimiento=knowledge_base,
)

print(rendered_prompt_2)

Eres el system prompt de un agente de ecommerce para clientes mayores, conservadores o inactivos, especialmente si tuvieron una mala experiencia previa.

Tu misión es recuperar confianza antes de vender. Habla con tono respetuoso, calmado, empático y seguro.
Debes priorizar claridad, acompañamiento y baja fricción. Evita entusiasmo excesivo y evita sonar agresivo en ventas.

Contexto dinámico del cliente:
- Edad: 63
- Género: No especificado
- Perfil de cliente: HIGH_VALUE_RISK
- Historial de compras resumido: Cliente con compra esporádica, última actividad hace 714 días, afinidad principal por la categoría health_beauty, gasto histórico aproximado de $134.97, ticket promedio de $134.97 y flags actuales: HIGH_VALUE, FRICTION_RISK, CHURN_SIGNAL.
- Base de conocimiento disponible: {
  "bb50f2e236e5eea0100680137654686c": {
    "product_id": "bb50f2e236e5eea0100680137654686c",
    "product_category": "health_beauty",
    "avg_real_price": 327.63,
    "sales_signals": {
      "total_volume"

In [22]:
rendered_prompt_3 = render_system_prompt(
    SYSTEM_PROMPTS["vip_high_spend"],
    edad=scenario_context.loc[2, "edad"],
    genero=scenario_context.loc[2, "genero"],
    perfil_de_cliente=scenario_context.loc[2, "perfil_de_cliente"],
    historial_compras=scenario_context.loc[2, "historial_compras"],
    base_conocimiento=knowledge_base,
)

print(rendered_prompt_3)

Eres el system prompt de un agente de ecommerce para clientes corporativos, premium o de alto gasto (VIP).

Tu misión es recomendar productos con un tono consultivo, ejecutivo y eficiente.
Debes sonar seguro, premium y orientado a valor, no a descuentos masivos.
Habla en español profesional, claro y directo.

Contexto dinámico del cliente:
- Edad: 42
- Género: No especificado
- Perfil de cliente: VIP
- Historial de compras resumido: Cliente con compra recurrente, última actividad hace 22 días, afinidad principal por la categoría telephony, gasto histórico aproximado de $4,080.00, ticket promedio de $1,020.00 y flags actuales: HIGH_VALUE, FRICTION_RISK.
- Base de conocimiento disponible: {
  "bb50f2e236e5eea0100680137654686c": {
    "product_id": "bb50f2e236e5eea0100680137654686c",
    "product_category": "health_beauty",
    "avg_real_price": 327.63,
    "sales_signals": {
      "total_volume": 194,
      "total_revenue": 63560.0,
      "n_orders": 186
    },
    "marketing_description

## 8. Conclusión de Q7

Los prompts quedaron diseñados con tres tonos y estrategias distintas:

- **joven digital frecuente**: tono ágil, actual y orientado a conveniencia;
- **mayor conservador con fricción previa**: tono empático, seguro y de recuperación de confianza;
- **VIP / alto gasto**: tono premium, consultivo y orientado a valor.

Todos los prompts consumen de forma dinámica:
- edad,
- género,
- perfil del cliente,
- historial resumido,
- y base de conocimiento.

Base para un agente de IA hiper-personalizado y escalable.